<a href="https://colab.research.google.com/github/simonburner/hands-on-LLM/blob/main/ch4/movie_reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datasets import load_dataset

In [2]:
data = load_dataset("cornell-movie-review-data/rotten_tomatoes")

In [3]:
data

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [4]:
data["train"][0, -1]

{'text': ['the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
  'things really get weird , though not particularly scary : the movie is all portent and no content .'],
 'label': [1, 0]}

In [5]:
from transformers import pipeline

In [6]:
model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

In [7]:
pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    return_all_scores=True,
    device="cuda:0"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
import numpy as np
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset

In [13]:
label_map = {"negative": 0, "neutral": 0, "positive": 1}

y_pred = []

for output in tqdm(pipe(KeyDataset(data["test"], "text")), total=len(data["test"])):
  y_pred.append(label_map[output["label"]])

100%|██████████| 1066/1066 [00:24<00:00, 43.30it/s]


In [14]:
from sklearn.metrics import classification_report

In [15]:
def evaluate_performance(y_true, y_pred):
  performance = classification_report(y_true,
                                      y_pred,
                                      target_names=["Negative Reviews", "Positive Reviews"]
  )
  print(performance)

In [16]:
evaluate_performance(data["test"]["label"], y_pred)

                  precision    recall  f1-score   support

Negative Reviews       0.68      0.94      0.79       533
Positive Reviews       0.91      0.56      0.69       533

        accuracy                           0.75      1066
       macro avg       0.79      0.75      0.74      1066
    weighted avg       0.79      0.75      0.74      1066

